<a href="https://colab.research.google.com/github/Abhijith311kumar/snapchat-streak-notification-ab-test/blob/main/notebooks/01_streak_notification_ab_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Installing DB and importing pandas

In [2]:
!pip install duckdb -q
import numpy as np, pandas as pd, duckdb
from scipy import stats

2. Data Simulation - All data is synthetic. I plant a known +2% lift in snaps per day and a small increase in notification opt-outs, so I can later check that my methods find them.
Have a rng to generate snaps per user per day having a slight higher IOS users over android and the option to opt-out of notifications

In [3]:
rng = np.random.default_rng(42)
n = 10000

users = pd.DataFrame({
    "user_id": range(n),
    "platform": rng.choice(["iOS", "Android"], n, p=[0.6, 0.4]),
    "country": rng.choice(["US", "UK", "IN", "BR"], n),
    "variant": rng.choice(["control", "treatment"], n),
})

# each user's "normal" snaps per day (heavy and light senders)
baseline = rng.gamma(2, 2, n)

# behavior BEFORE the test (you will use this later for CUPED)
users["pre_snaps"] = rng.poisson(baseline)

# behavior DURING the test, with a planted +2% lift for treatment
lift = np.where(users.variant == "treatment", 1.02, 1.0)
users["snaps_per_day"] = rng.poisson(baseline * lift)

# guardrail: treatment users opt out of notifications slightly more
p_optout = np.where(users.variant == "treatment", 0.045, 0.030)
users["opted_out"] = rng.binomial(1, p_optout)

users.head()

,user_id,platform,country,variant,pre_snaps,snaps_per_day,opted_out
0,0,Android,US,control,6,6,0
1,1,iOS,IN,control,11,13,0
2,2,Android,US,treatment,5,7,0
3,3,Android,IN,control,2,4,0
4,4,iOS,IN,control,0,1,0


3. Load into DB and run my query to show number if users in each variant who have opted out from notifications and shows avg snaps per day.

In [4]:
duckdb.sql("""
    SELECT variant,
           COUNT(*)            AS users,
           AVG(snaps_per_day)  AS avg_snaps,
           AVG(opted_out)      AS optout_rate
    FROM users
    GROUP BY variant
""").show()

┌───────────┬───────┬────────────────────┬─────────────────────┐
│  variant  │ users │     avg_snaps      │     optout_rate     │
│  varchar  │ int64 │       double       │       double        │
├───────────┼───────┼────────────────────┼─────────────────────┤
│ treatment │  5028 │ 4.0562848050914875 │  0.0405727923627685 │
│ control   │  4972 │  4.057723250201127 │ 0.02835880933226066 │
└───────────┴───────┴────────────────────┴─────────────────────┘



4. A/B Testing

In [5]:
c = users[users.variant == "control"].snaps_per_day
t = users[users.variant == "treatment"].snaps_per_day

t_stat, p_value = stats.ttest_ind(t, c, equal_var=False)
lift_pct = (t.mean() / c.mean() - 1) * 100
print(f"Lift: {lift_pct:.2f}%  |  p-value: {p_value:.3f}")

Lift: -0.04%  |  p-value: 0.984


# Analysis
The results came back showing a -0.04% dip for treatment ($p = 0.984$)—so basically no detectable difference. But since this is a simulation where we know there's a real +2% lift hiding in there, this is a classic false negative. With only 10k users and a ton of natural variance in snap counts, the test just didn't have enough statistical power. Next up, I'm going to run a power analysis to figure out our required sample size and apply CUPED to help cut through the noise.